In [1]:
import pandas as pd

df = pd.read_csv("data/weatherAUS.csv")
print(df.shape)
df.head()


(145460, 23)


,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 145460 entries, 0 to 145459
Data columns (total 23 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Date           145460 non-null  str    
 1   Location       145460 non-null  str    
 2   MinTemp        143975 non-null  float64
 3   MaxTemp        144199 non-null  float64
 4   Rainfall       142199 non-null  float64
 5   Evaporation    82670 non-null   float64
 6   Sunshine       75625 non-null   float64
 7   WindGustDir    135134 non-null  str    
 8   WindGustSpeed  135197 non-null  float64
 9   WindDir9am     134894 non-null  str    
 10  WindDir3pm     141232 non-null  str    
 11  WindSpeed9am   143693 non-null  float64
 12  WindSpeed3pm   142398 non-null  float64
 13  Humidity9am    142806 non-null  float64
 14  Humidity3pm    140953 non-null  float64
 15  Pressure9am    130395 non-null  float64
 16  Pressure3pm    130432 non-null  float64
 17  Cloud9am       89572 non-null   float64


In [3]:
df.isna().mean().sort_values(ascending=False) * 100

Sunshine         48.009762
Evaporation      43.166506
Cloud3pm         40.807095
Cloud9am         38.421559
Pressure9am      10.356799
Pressure3pm      10.331363
WindDir9am        7.263853
WindGustDir       7.098859
WindGustSpeed     7.055548
Humidity3pm       3.098446
WindDir3pm        2.906641
Temp3pm           2.481094
RainTomorrow      2.245978
Rainfall          2.241853
RainToday         2.241853
WindSpeed3pm      2.105046
Humidity9am       1.824557
WindSpeed9am      1.214767
Temp9am           1.214767
MinTemp           1.020899
MaxTemp           0.866905
Date              0.000000
Location          0.000000
dtype: float64

In [4]:
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Location", "Date"]).reset_index(drop=True)
df[["Date", "Location", "Rainfall"]].head()

,Date,Location,Rainfall
0,2008-07-01,Adelaide,5.0
1,2008-07-02,Adelaide,0.8
2,2008-07-03,Adelaide,0.0
3,2008-07-04,Adelaide,0.0
4,2008-07-05,Adelaide,0.0


In [5]:
df["RainfallTomorrow"] = df.groupby("Location")["Rainfall"].shift(-1)

next_date = df.groupby("Location")["Date"].shift(-1)
gap = (next_date - df["Date"]).dt.days
df.loc[gap != 1, "RainfallTomorrow"] = None

df[["Location", "Date", "Rainfall", "RainfallTomorrow"]].head(10)

,Location,Date,Rainfall,RainfallTomorrow
0,Adelaide,2008-07-01,5.0,0.8
1,Adelaide,2008-07-02,0.8,0.0
2,Adelaide,2008-07-03,0.0,0.0
3,Adelaide,2008-07-04,0.0,0.0
4,Adelaide,2008-07-05,0.0,NaN
5,Adelaide,2008-07-06,NaN,16.2
6,Adelaide,2008-07-07,16.2,17.0
7,Adelaide,2008-07-08,17.0,1.8
8,Adelaide,2008-07-09,1.8,9.0
9,Adelaide,2008-07-10,9.0,0.2


In [6]:
df = df.dropna(subset=["RainfallTomorrow"])
df["HeavyRainTomorrow"] = (df["RainfallTomorrow"] > 64.5).astype(int)

print(df.shape)
df["HeavyRainTomorrow"].value_counts()

(142017, 25)


HeavyRainTomorrow
0    141554
1       463
Name: count, dtype: int64

In [7]:
df = df.drop(columns=["RainTomorrow", "RainfallTomorrow"])
df["Month"] = df["Date"].dt.month
df.head()

,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,HeavyRainTomorrow,Month
0,2008-07-01,Adelaide,8.8,15.7,5.0,1.6,2.6,NW,48.0,SW,...,67.0,1017.4,1017.7,NaN,NaN,13.5,14.9,Yes,0,7
1,2008-07-02,Adelaide,12.7,15.8,0.8,1.4,7.8,SW,35.0,SSW,...,52.0,1022.4,1022.6,NaN,NaN,13.7,15.5,No,0,7
2,2008-07-03,Adelaide,6.2,15.1,0.0,1.8,2.1,W,20.0,NNE,...,56.0,1027.8,1026.5,NaN,NaN,9.3,13.9,No,0,7
3,2008-07-04,Adelaide,5.3,15.9,0.0,1.4,8.0,NNE,30.0,NNE,...,46.0,1028.7,1025.6,NaN,NaN,10.2,15.3,No,0,7
5,2008-07-06,Adelaide,11.3,15.7,NaN,NaN,1.5,NNW,52.0,NNE,...,62.0,1019.5,1016.2,NaN,NaN,13.0,14.4,NaN,0,7


In [8]:
train = df[df["Date"] < "2015-01-01"]
val   = df[(df["Date"] >= "2015-01-01") & (df["Date"] < "2016-01-01")]
test  = df[df["Date"] >= "2016-01-01"]

print("Train:", train.shape, " Heavy days:", train["HeavyRainTomorrow"].sum())
print("Val:  ", val.shape,   " Heavy days:", val["HeavyRainTomorrow"].sum())
print("Test: ", test.shape,  " Heavy days:", test["HeavyRainTomorrow"].sum())

Train: (98853, 24)  Heavy days: 319
Val:   (17231, 24)  Heavy days: 58
Test:  (25933, 24)  Heavy days: 86


In [9]:
target = "HeavyRainTomorrow"

X_train, y_train = train.drop(columns=["Date", target]), train[target]
X_val,   y_val   = val.drop(columns=["Date", target]),   val[target]
X_test,  y_test  = test.drop(columns=["Date", target]),  test[target]

print(X_train.shape, y_train.shape)

(98853, 22) (98853,)


In [10]:
num_cols = ["MinTemp", "MaxTemp", "Rainfall", "Evaporation", "Sunshine",
            "WindGustSpeed", "WindSpeed9am", "WindSpeed3pm",
            "Humidity9am", "Humidity3pm", "Pressure9am", "Pressure3pm",
            "Cloud9am", "Cloud3pm", "Temp9am", "Temp3pm", "Month"]

cat_cols = ["Location", "WindGustDir", "WindDir9am", "WindDir3pm", "RainToday"]

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_pipe = Pipeline([
    ("fill",  SimpleImputer(strategy="median", add_indicator=True)),
    ("scale", StandardScaler())
])

cat_pipe = Pipeline([
    ("fill",   SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

prep = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

In [12]:
import numpy as np

prep.fit(X_train)                  # learn medians, scales, categories from TRAIN only
X_train_ready = prep.transform(X_train)
X_val_ready   = prep.transform(X_val)

print("Before:", X_train.shape)
print("After: ", X_train_ready.shape)
print("Missing values left:", np.isnan(X_train_ready).sum())

Before: (98853, 22)
After:  (98853, 132)
Missing values left: 0


In [1]:
from src.preprocessing import load_and_clean, time_split, build_preprocessor, BASE_NUM, TARGET

df2 = load_and_clean()
train, val, test = time_split(df2)

print("Rows after cleaning:", df2.shape)
print("Train / Val / Test:", len(train), len(val), len(test))
print("Heavy-rain days:", df2[TARGET].sum())

Rows after cleaning: (142017, 24)
Train / Val / Test: 98853 17231 25933
Heavy-rain days: 463
